# Exploración y extracción · API-Football

**Competición:** Copa Mundial (`league = 1`)  
**Temporada:** 2022  
**Elaborado por:** emanuel

Este notebook documenta paso a paso el proceso que transforma las respuestas
JSON de `/teams`, `/fixtures` y `/standings` en tres archivos Parquet limpios.

Cabe aclarar que el archivo `extractor_api.py` hace la extraccion tambien y lo cree porque se pide textual en el pdf de instrucciones, pero me gusta mas hacer la exploracion de datos en un notebook por lo visual que se vuelve (culpa de juanse).

> La clave se lee desde `.env` y nunca se muestra. La ejecución normal reutiliza
> la caché para proteger la cuota diaria de API-Football.

## 1. Preparación

Se importan las librerías requeridas y se fijan los parámetros del ejercicio.
`ACTUALIZAR = False` significa que se usará la caché si ya existe. Solo debe
cambiarse a `True` cuando se quiera volver a consultar la API.

In [16]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os

import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

BASE_URL = "https://v3.football.api-sports.io"
COMPETENCIA_ID = 1
TEMPORADA = 2022
EXTRAIDO_POR = "emanuel"
DIRECTORIO = Path.cwd()
CACHE_DIR = DIRECTORIO / "cache"
ACTUALIZAR = False

print(f"Proyecto: league={COMPETENCIA_ID}, season={TEMPORADA}")
print(f"Directorio de trabajo: {DIRECTORIO}")

Proyecto: league=1, season=2022
Directorio de trabajo: /home/emanuel/Universidad/Sexto/Ingenieria de Datos/Taller1INGDATA/punto2/apis


### Lectura segura de la credencial

La clave se carga con `python-dotenv`. Solo se comprueba que exista; su contenido
no se imprime ni se guarda en el notebook.

In [17]:
load_dotenv(DIRECTORIO / ".env")
api_key = os.getenv("API_SPORTS_KEY", "").strip()
if not api_key:
    raise RuntimeError("Falta API_SPORTS_KEY en el archivo .env")
print("Credencial cargada correctamente (valor oculto).")

Credencial cargada correctamente (valor oculto).


## 2. Función de consulta, validación y caché

Para cada endpoint, la función:

1. Busca una respuesta JSON guardada en `cache/`.
2. Si no existe, hace una solicitud GET con `requests`.
3. Verifica el estado HTTP y el campo `errors` de API-Football.
4. Revisa `paging.current` y `paging.total`.
5. Recorre páginas adicionales cuando sean necesarias.
6. Conserva cada página válida para evitar llamadas repetidas.

In [18]:
def validar_respuesta(cuerpo, endpoint):
    if not isinstance(cuerpo, dict):
        raise RuntimeError(f"/{endpoint}: la respuesta no es un objeto JSON")
    if cuerpo.get("errors"):
        raise RuntimeError(f"/{endpoint}: {cuerpo['errors']}")
    if not isinstance(cuerpo.get("response"), list):
        raise RuntimeError(f"/{endpoint}: falta la lista response")
    if not cuerpo["response"]:
        raise RuntimeError(f"/{endpoint}: la consulta no devolvió registros")
    paginacion = cuerpo.get("paging")
    if not isinstance(paginacion, dict):
        raise RuntimeError(f"/{endpoint}: falta la información de paginación")
    actual = int(paginacion.get("current", 0))
    total = int(paginacion.get("total", 0))
    if actual < 1 or total < actual:
        raise RuntimeError(f"/{endpoint}: paginación incoherente {paginacion}")
    return cuerpo


def consultar_endpoint(endpoint, actualizar=False):
    CACHE_DIR.mkdir(exist_ok=True)
    registros = []
    cuerpos = []
    trazabilidad = []
    pagina = 1
    total_paginas = 1

    with requests.Session() as sesion:
        sesion.headers.update({"x-apisports-key": api_key})
        while pagina <= total_paginas:
            archivo_cache = CACHE_DIR / f"{endpoint}_pagina_{pagina}.json"
            if archivo_cache.exists() and not actualizar:
                cuerpo = json.loads(archivo_cache.read_text(encoding="utf-8"))
                origen = "caché"
            else:
                parametros = {"league": COMPETENCIA_ID, "season": TEMPORADA}
                # Estos endpoints no aceptan page si no existe una segunda página.
                if pagina > 1:
                    parametros["page"] = pagina
                respuesta_http = sesion.get(
                    f"{BASE_URL}/{endpoint}",
                    headers={"x-apisports-key": api_key},
                    params=parametros,
                    timeout=30,
                )
                respuesta_http.raise_for_status()
                cuerpo = respuesta_http.json()
                validar_respuesta(cuerpo, endpoint)
                temporal = archivo_cache.with_suffix(".tmp")
                temporal.write_text(
                    json.dumps(cuerpo, ensure_ascii=False, indent=2),
                    encoding="utf-8",
                )
                temporal.replace(archivo_cache)
                origen = "API"

            cuerpo = validar_respuesta(cuerpo, endpoint)
            total_paginas = int(cuerpo["paging"]["total"])
            registros.extend(cuerpo["response"])
            cuerpos.append(cuerpo)
            trazabilidad.append({
                "endpoint": f"/{endpoint}",
                "pagina": pagina,
                "total_paginas": total_paginas,
                "registros_pagina": len(cuerpo["response"]),
                "origen": origen,
            })
            pagina += 1

    return registros, cuerpos, trazabilidad

## 3. Extracción de los tres endpoints

In [19]:
equipos_json, cuerpos_equipos, traza_equipos = consultar_endpoint("teams", ACTUALIZAR)
partidos_json, cuerpos_partidos, traza_partidos = consultar_endpoint("fixtures", ACTUALIZAR)
standings_json, cuerpos_standings, traza_standings = consultar_endpoint("standings", ACTUALIZAR)

trazabilidad = pd.DataFrame(traza_equipos + traza_partidos + traza_standings)
display(trazabilidad.style.hide(axis="index").set_caption("Resumen de la extracción"))

endpoint,pagina,total_paginas,registros_pagina,origen
/teams,1,1,32,caché
/fixtures,1,1,64,caché
/standings,1,1,1,caché


## 4. Exploración de las respuestas JSON

Antes de construir tablas, se inspeccionan las rutas y tipos de datos presentes
en un registro representativo. Esta exploración permite descubrir los niveles
anidados sin escribir manualmente nombres de equipos, estadios o resultados.

In [20]:
def rutas_json(objeto, prefijo=""):
    # Devuelve las rutas y tipos de un objeto JSON representativo.
    filas = []
    if isinstance(objeto, dict):
        for clave, valor in objeto.items():
            ruta = f"{prefijo}.{clave}" if prefijo else clave
            filas.append({"ruta_json": ruta, "tipo": type(valor).__name__})
            filas.extend(rutas_json(valor, ruta))
    elif isinstance(objeto, list) and objeto:
        filas.extend(rutas_json(objeto[0], f"{prefijo}[]"))
    return filas


resumen_crudo = pd.DataFrame([
    {"endpoint": "/teams", "registros": len(equipos_json), "claves_raiz": ", ".join(equipos_json[0])},
    {"endpoint": "/fixtures", "registros": len(partidos_json), "claves_raiz": ", ".join(partidos_json[0])},
    {"endpoint": "/standings", "registros": len(standings_json), "claves_raiz": ", ".join(standings_json[0])},
])
display(resumen_crudo.style.hide(axis="index").set_caption("Contenido crudo recibido"))

endpoint,registros,claves_raiz
/teams,32,"team, venue"
/fixtures,64,"fixture, league, teams, goals, score"
/standings,1,league


### 4.1 Exploración de `/teams`

In [21]:
estructura_equipos = pd.DataFrame(rutas_json(equipos_json[0])).drop_duplicates()
display(estructura_equipos.style.hide(axis="index").set_caption("Rutas encontradas en un equipo"))

vista_equipos = pd.json_normalize(equipos_json, sep=".")
columnas_equipo_interes = [c for c in vista_equipos.columns if c.startswith("team.")]
display(vista_equipos[columnas_equipo_interes].head(5))

ruta_json,tipo
team,dict
team.id,int
team.name,str
team.code,str
team.country,str
team.founded,int
team.national,bool
team.logo,str
venue,dict
venue.id,int


,team.id,team.name,team.code,team.country,team.founded,team.national,team.logo
0,1,Belgium,BEL,Belgium,1895,True,https://media.api-sports.io/football/teams/1.png
1,2,France,FRA,France,1919,True,https://media.api-sports.io/football/teams/2.png
2,3,Croatia,CRO,Croatia,1912,True,https://media.api-sports.io/football/teams/3.png
3,6,Brazil,BRA,Brazil,1914,True,https://media.api-sports.io/football/teams/6.png
4,7,Uruguay,URU,Uruguay,1900,True,https://media.api-sports.io/football/teams/7.png


### 4.2 Exploración de `/fixtures`

In [22]:
estructura_partidos = pd.DataFrame(rutas_json(partidos_json[0])).drop_duplicates()
display(estructura_partidos.style.hide(axis="index").set_caption("Rutas encontradas en un partido"))

vista_partidos = pd.json_normalize(partidos_json, sep=".")
columnas_partido_interes = [
    "fixture.id", "fixture.date", "league.round",
    "teams.home.name", "goals.home", "goals.away", "teams.away.name",
    "fixture.venue.name",
]
display(vista_partidos[columnas_partido_interes].head(8))

ruta_json,tipo
fixture,dict
fixture.id,int
fixture.referee,str
fixture.timezone,str
fixture.date,str
fixture.timestamp,int
fixture.periods,dict
fixture.periods.first,int
fixture.periods.second,int
fixture.venue,dict


,fixture.id,fixture.date,league.round,teams.home.name,goals.home,goals.away,teams.away.name,fixture.venue.name
0,855736,2022-11-20T16:00:00+00:00,Group Stage - 1,Qatar,0,2,Ecuador,Al Bayt Stadium
1,855735,2022-11-21T13:00:00+00:00,Group Stage - 1,England,6,2,Iran,Khalifa International Stadium
2,855734,2022-11-21T16:00:00+00:00,Group Stage - 1,Senegal,0,2,Netherlands,Al Thumama Stadium
3,866681,2022-11-21T19:00:00+00:00,Group Stage - 1,USA,1,1,Wales,Ahmad bin Ali Stadium
4,855737,2022-11-22T10:00:00+00:00,Group Stage - 1,Argentina,1,2,Saudi Arabia,Lusail Iconic Stadium
5,855738,2022-11-22T13:00:00+00:00,Group Stage - 1,Denmark,0,0,Tunisia,Education City Stadium
6,855739,2022-11-22T16:00:00+00:00,Group Stage - 1,Mexico,0,0,Poland,Stadium 974
7,871850,2022-11-22T19:00:00+00:00,Group Stage - 1,France,4,1,Australia,Al Janoub Stadium


### 4.3 Exploración de `/standings`

In [23]:
liga_standings = standings_json[0]["league"]
grupos_json = liga_standings["standings"]
filas_standings_json = [fila for grupo in grupos_json for fila in grupo]

print(f"Grupos encontrados: {len(grupos_json)}")
print(f"Equipos encontrados en standings: {len(filas_standings_json)}")
estructura_standings = pd.DataFrame(rutas_json(filas_standings_json[0])).drop_duplicates()
display(estructura_standings.style.hide(axis="index").set_caption("Rutas de una fila de clasificación"))

vista_standings = pd.json_normalize(filas_standings_json, sep=".")
display(vista_standings[[
    "group", "rank", "team.name", "points", "all.played",
    "all.win", "all.draw", "all.lose", "goalsDiff",
]].head(8))

Grupos encontrados: 8
Equipos encontrados en standings: 32


ruta_json,tipo
rank,int
team,dict
team.id,int
team.name,str
team.logo,str
points,int
goalsDiff,int
group,str
form,str
status,str


,group,rank,team.name,points,all.played,all.win,all.draw,all.lose,goalsDiff
0,Group A,1,Netherlands,7,3,2,1,0,4
1,Group A,2,Senegal,6,3,2,0,1,1
2,Group A,3,Ecuador,4,3,1,1,1,1
3,Group A,4,Qatar,0,3,0,0,3,-6
4,Group B,1,England,7,3,2,1,0,7
5,Group B,2,USA,5,3,1,2,0,1
6,Group B,3,Iran,3,3,1,0,2,-3
7,Group B,4,Wales,1,3,0,1,2,-5


## 5. Diseño de la normalización

`pandas.json_normalize` convierte las rutas anidadas en columnas separadas por
puntos. Luego se seleccionan y renombran solo los campos pedidos, se agregan los
metadatos del proceso y se asignan tipos anulables de pandas.

In [24]:
def convertir_enteros(df, columnas):
    for columna in columnas:
        df[columna] = pd.to_numeric(df[columna], errors="coerce").astype("Int64")


def convertir_textos(df, columnas):
    for columna in columnas:
        df[columna] = df[columna].astype("string")


fecha_extraccion = datetime.now(timezone.utc)

### 5.1 Normalización de equipos

In [25]:
mapa_equipos = {
    "team.id": "equipo_id",
    "team.name": "nombre_equipo",
    "team.code": "codigo_equipo",
    "team.country": "pais",
    "team.founded": "anio_fundacion",
    "team.national": "es_seleccion_nacional",
    "team.logo": "logo_url",
}
equipos = (
    pd.json_normalize(equipos_json, sep=".")
    .reindex(columns=mapa_equipos)
    .rename(columns=mapa_equipos)
)
equipos = equipos.assign(
    competencia_id=COMPETENCIA_ID,
    temporada=TEMPORADA,
    fecha_extraccion=fecha_extraccion,
    extraido_por=EXTRAIDO_POR,
    endpoint_origen="/teams",
)
columnas_equipos = [
    "equipo_id", "nombre_equipo", "codigo_equipo", "pais", "anio_fundacion",
    "es_seleccion_nacional", "logo_url", "competencia_id", "temporada",
    "fecha_extraccion", "extraido_por", "endpoint_origen",
]
equipos = equipos[columnas_equipos]
convertir_enteros(equipos, ["equipo_id", "anio_fundacion", "competencia_id", "temporada"])
convertir_textos(equipos, ["nombre_equipo", "codigo_equipo", "pais", "logo_url", "extraido_por", "endpoint_origen"])
equipos["es_seleccion_nacional"] = equipos["es_seleccion_nacional"].astype("boolean")
equipos["fecha_extraccion"] = pd.to_datetime(equipos["fecha_extraccion"], utc=True)
equipos = equipos.drop_duplicates("equipo_id").sort_values("nombre_equipo").reset_index(drop=True)
display(equipos.head())

,equipo_id,nombre_equipo,codigo_equipo,pais,anio_fundacion,es_seleccion_nacional,logo_url,competencia_id,temporada,fecha_extraccion,extraido_por,endpoint_origen
0,26,Argentina,ARG,Argentina,1893,True,https://media.api-sports.io/football/teams/26.png,1,2022,2026-08-17 23:28:26.702764+00:00,emanuel,/teams
1,20,Australia,AUS,Australia,1961,True,https://media.api-sports.io/football/teams/20.png,1,2022,2026-08-17 23:28:26.702764+00:00,emanuel,/teams
2,1,Belgium,BEL,Belgium,1895,True,https://media.api-sports.io/football/teams/1.png,1,2022,2026-08-17 23:28:26.702764+00:00,emanuel,/teams
3,6,Brazil,BRA,Brazil,1914,True,https://media.api-sports.io/football/teams/6.png,1,2022,2026-08-17 23:28:26.702764+00:00,emanuel,/teams
4,1530,Cameroon,CAM,Cameroon,1959,True,https://media.api-sports.io/football/teams/1530.png,1,2022,2026-08-17 23:28:26.702764+00:00,emanuel,/teams


### 5.2 Normalización de partidos

In [26]:
mapa_partidos = {
    "fixture.id": "partido_id",
    "league.id": "competencia_id",
    "league.name": "competencia_nombre",
    "league.season": "temporada",
    "league.round": "ronda",
    "fixture.date": "fecha_partido",
    "fixture.timezone": "zona_horaria",
    "fixture.status.long": "estado_partido",
    "fixture.status.elapsed": "minuto_transcurrido",
    "fixture.referee": "arbitro",
    "fixture.venue.id": "estadio_id",
    "fixture.venue.name": "estadio_nombre",
    "fixture.venue.city": "estadio_ciudad",
    "teams.home.id": "equipo_local_id",
    "teams.home.name": "equipo_local_nombre",
    "teams.away.id": "equipo_visitante_id",
    "teams.away.name": "equipo_visitante_nombre",
    "teams.home.winner": "gano_local",
    "teams.away.winner": "gano_visitante",
    "goals.home": "goles_local",
    "goals.away": "goles_visitante",
    "score.penalty.home": "penales_local",
    "score.penalty.away": "penales_visitante",
}
partidos = (
    pd.json_normalize(partidos_json, sep=".")
    .reindex(columns=mapa_partidos)
    .rename(columns=mapa_partidos)
)
partidos = partidos.assign(
    fecha_extraccion=fecha_extraccion,
    extraido_por=EXTRAIDO_POR,
    endpoint_origen="/fixtures",
)
columnas_partidos = [
    "partido_id", "competencia_id", "competencia_nombre", "temporada", "ronda",
    "fecha_partido", "zona_horaria", "estado_partido", "minuto_transcurrido",
    "arbitro", "estadio_id", "estadio_nombre", "estadio_ciudad", "equipo_local_id",
    "equipo_local_nombre", "equipo_visitante_id", "equipo_visitante_nombre",
    "gano_local", "gano_visitante", "goles_local", "goles_visitante",
    "penales_local", "penales_visitante", "fecha_extraccion", "extraido_por",
    "endpoint_origen",
]
partidos = partidos[columnas_partidos]
convertir_enteros(partidos, [
    "partido_id", "competencia_id", "temporada", "minuto_transcurrido", "estadio_id",
    "equipo_local_id", "equipo_visitante_id", "goles_local", "goles_visitante",
    "penales_local", "penales_visitante",
])
convertir_textos(partidos, [
    "competencia_nombre", "ronda", "zona_horaria", "estado_partido", "arbitro",
    "estadio_nombre", "estadio_ciudad", "equipo_local_nombre",
    "equipo_visitante_nombre", "extraido_por", "endpoint_origen",
])
partidos["gano_local"] = partidos["gano_local"].fillna(False).astype("boolean")
partidos["gano_visitante"] = partidos["gano_visitante"].fillna(False).astype("boolean")
partidos["fecha_partido"] = pd.to_datetime(partidos["fecha_partido"], errors="coerce", utc=True)
partidos["fecha_extraccion"] = pd.to_datetime(partidos["fecha_extraccion"], utc=True)
partidos = partidos.drop_duplicates("partido_id").sort_values("fecha_partido").reset_index(drop=True)
display(partidos.head())

,partido_id,competencia_id,competencia_nombre,temporada,ronda,fecha_partido,zona_horaria,estado_partido,minuto_transcurrido,arbitro,estadio_id,estadio_nombre,estadio_ciudad,equipo_local_id,equipo_local_nombre,equipo_visitante_id,equipo_visitante_nombre,gano_local,gano_visitante,goles_local,goles_visitante,penales_local,penales_visitante,fecha_extraccion,extraido_por,endpoint_origen
0,855736,1,World Cup,2022,Group Stage - 1,2022-11-20 16:00:00+00:00,UTC,Match Finished,90,D. Orsato,<NA>,Al Bayt Stadium,Al Khor,1569,Qatar,2382,Ecuador,False,True,0,2,<NA>,<NA>,2026-08-17 23:28:26.702764+00:00,emanuel,/fixtures
1,855735,1,World Cup,2022,Group Stage - 1,2022-11-21 13:00:00+00:00,UTC,Match Finished,90,Raphael Claus,22430,Khalifa International Stadium,Ar-Rayyan,10,England,22,Iran,True,False,6,2,<NA>,<NA>,2026-08-17 23:28:26.702764+00:00,emanuel,/fixtures
2,855734,1,World Cup,2022,Group Stage - 1,2022-11-21 16:00:00+00:00,UTC,Match Finished,90,Wilton Pereira Sampaio,<NA>,Al Thumama Stadium,Doha,13,Senegal,1118,Netherlands,False,True,0,2,<NA>,<NA>,2026-08-17 23:28:26.702764+00:00,emanuel,/fixtures
3,866681,1,World Cup,2022,Group Stage - 1,2022-11-21 19:00:00+00:00,UTC,Match Finished,90,Abdulrahman Al Jassim,<NA>,Ahmad bin Ali Stadium,Al-Rayyan,2384,USA,767,Wales,False,False,1,1,<NA>,<NA>,2026-08-17 23:28:26.702764+00:00,emanuel,/fixtures
4,855737,1,World Cup,2022,Group Stage - 1,2022-11-22 10:00:00+00:00,UTC,Match Finished,90,S. Vinčić,<NA>,Lusail Iconic Stadium,Lusail,26,Argentina,23,Saudi Arabia,False,True,1,2,<NA>,<NA>,2026-08-17 23:28:26.702764+00:00,emanuel,/fixtures


### 5.3 Normalización de la clasificación

In [27]:
mapa_clasificacion = {
    "group": "grupo",
    "rank": "posicion",
    "team.id": "equipo_id",
    "team.name": "nombre_equipo",
    "points": "puntos",
    "all.played": "partidos_jugados",
    "all.win": "partidos_ganados",
    "all.draw": "partidos_empatados",
    "all.lose": "partidos_perdidos",
    "all.goals.for": "goles_favor",
    "all.goals.against": "goles_contra",
    "goalsDiff": "diferencia_gol",
    "form": "forma_reciente",
    "status": "estado_clasificacion",
    "description": "descripcion_clasificacion",
    "update": "fecha_actualizacion",
}
clasificacion = (
    pd.json_normalize(filas_standings_json, sep=".")
    .reindex(columns=mapa_clasificacion)
    .rename(columns=mapa_clasificacion)
)
clasificacion = clasificacion.assign(
    competencia_id=liga_standings.get("id", COMPETENCIA_ID),
    temporada=liga_standings.get("season", TEMPORADA),
    fecha_extraccion=fecha_extraccion,
    extraido_por=EXTRAIDO_POR,
    endpoint_origen="/standings",
)
columnas_clasificacion = [
    "grupo", "posicion", "equipo_id", "nombre_equipo", "puntos",
    "partidos_jugados", "partidos_ganados", "partidos_empatados",
    "partidos_perdidos", "goles_favor", "goles_contra", "diferencia_gol",
    "forma_reciente", "estado_clasificacion", "descripcion_clasificacion",
    "fecha_actualizacion", "competencia_id", "temporada", "fecha_extraccion",
    "extraido_por", "endpoint_origen",
]
clasificacion = clasificacion[columnas_clasificacion]
convertir_enteros(clasificacion, [
    "posicion", "equipo_id", "puntos", "partidos_jugados", "partidos_ganados",
    "partidos_empatados", "partidos_perdidos", "goles_favor", "goles_contra",
    "diferencia_gol", "competencia_id", "temporada",
])
convertir_textos(clasificacion, [
    "grupo", "nombre_equipo", "forma_reciente", "estado_clasificacion",
    "descripcion_clasificacion", "extraido_por", "endpoint_origen",
])
clasificacion["fecha_actualizacion"] = pd.to_datetime(clasificacion["fecha_actualizacion"], errors="coerce", utc=True)
clasificacion["fecha_extraccion"] = pd.to_datetime(clasificacion["fecha_extraccion"], utc=True)
clasificacion = (
    clasificacion.drop_duplicates(["grupo", "equipo_id"])
    .sort_values(["grupo", "posicion"])
    .reset_index(drop=True)
)
display(clasificacion.head(8))

,grupo,posicion,equipo_id,nombre_equipo,puntos,partidos_jugados,partidos_ganados,partidos_empatados,partidos_perdidos,goles_favor,goles_contra,diferencia_gol,forma_reciente,estado_clasificacion,descripcion_clasificacion,fecha_actualizacion,competencia_id,temporada,fecha_extraccion,extraido_por,endpoint_origen
0,Group A,1,1118,Netherlands,7,3,2,1,0,5,1,4,WDW,same,Promotion - World Cup (Play Offs),2022-12-12 00:00:00+00:00,1,2022,2026-08-17 23:28:26.702764+00:00,emanuel,/standings
1,Group A,2,13,Senegal,6,3,2,0,1,5,4,1,WWL,same,Promotion - World Cup (Play Offs),2022-12-12 00:00:00+00:00,1,2022,2026-08-17 23:28:26.702764+00:00,emanuel,/standings
2,Group A,3,2382,Ecuador,4,3,1,1,1,4,3,1,LDW,same,<NA>,2022-12-12 00:00:00+00:00,1,2022,2026-08-17 23:28:26.702764+00:00,emanuel,/standings
3,Group A,4,1569,Qatar,0,3,0,0,3,1,7,-6,LLL,same,<NA>,2022-12-12 00:00:00+00:00,1,2022,2026-08-17 23:28:26.702764+00:00,emanuel,/standings
4,Group B,1,10,England,7,3,2,1,0,9,2,7,WDW,same,Promotion - World Cup (Play Offs),2022-12-12 00:00:00+00:00,1,2022,2026-08-17 23:28:26.702764+00:00,emanuel,/standings
5,Group B,2,2384,USA,5,3,1,2,0,2,1,1,WDD,same,Promotion - World Cup (Play Offs),2022-12-12 00:00:00+00:00,1,2022,2026-08-17 23:28:26.702764+00:00,emanuel,/standings
6,Group B,3,22,Iran,3,3,1,0,2,4,7,-3,LWL,same,<NA>,2022-12-12 00:00:00+00:00,1,2022,2026-08-17 23:28:26.702764+00:00,emanuel,/standings
7,Group B,4,767,Wales,1,3,0,1,2,1,6,-5,LLD,same,<NA>,2022-12-12 00:00:00+00:00,1,2022,2026-08-17 23:28:26.702764+00:00,emanuel,/standings


## 6. Validación antes de guardar

Se comprueba que cada tabla tenga el esquema exacto, claves no nulas y únicas,
y ninguna celda que conserve listas o diccionarios.

In [28]:
def validar_tabla(df, columnas, claves, nombre):
    controles = {
        "tiene_registros": not df.empty,
        "columnas_exactas": list(df.columns) == columnas,
        "claves_no_nulas": df[claves].notna().all(axis=None),
        "claves_unicas": not df.duplicated(claves).any(),
        "sin_listas_diccionarios": not df.map(lambda x: isinstance(x, (list, dict))).any(axis=None),
    }
    if not all(controles.values()):
        raise ValueError(f"Falló la validación de {nombre}: {controles}")
    return {"tabla": nombre, **controles, "filas": len(df), "columnas": df.shape[1]}


controles = pd.DataFrame([
    validar_tabla(equipos, columnas_equipos, ["equipo_id"], "equipos"),
    validar_tabla(partidos, columnas_partidos, ["partido_id"], "partidos"),
    validar_tabla(clasificacion, columnas_clasificacion, ["grupo", "equipo_id"], "clasificacion"),
])
display(controles.style.hide(axis="index").set_caption("Controles previos a la escritura"))

tabla,tiene_registros,columnas_exactas,claves_no_nulas,claves_unicas,sin_listas_diccionarios,filas,columnas
equipos,True,True,True,True,True,32,12
partidos,True,True,True,True,True,64,26
clasificacion,True,True,True,True,True,32,21


## 7. Escritura de los archivos Parquet

In [29]:
salidas = {
    "equipos.parquet": equipos,
    "partidos.parquet": partidos,
    "clasificacion.parquet": clasificacion,
}

for nombre, df in salidas.items():
    temporal = DIRECTORIO / f".{nombre}.tmp"
    destino = DIRECTORIO / nombre
    df.to_parquet(temporal, index=False, engine="pyarrow")
    temporal.replace(destino)

resumen_salidas = pd.DataFrame([
    {
        "archivo": nombre,
        "filas": len(df),
        "columnas": df.shape[1],
        "tamano_kb": round((DIRECTORIO / nombre).stat().st_size / 1024, 2),
    }
    for nombre, df in salidas.items()
])
display(resumen_salidas.style.hide(axis="index").set_caption("Archivos Parquet generados"))

archivo,filas,columnas,tamano_kb
equipos.parquet,32,12,8.910000
partidos.parquet,64,26,18.590000
clasificacion.parquet,32,21,14.040000


## 8. Verificación de lectura

Como comprobación final, los archivos se leen nuevamente desde disco. Esto
garantiza que el notebook de análisis podrá utilizarlos sin depender de las
variables creadas durante la extracción.

In [30]:
verificacion = []
for nombre in salidas:
    recuperado = pd.read_parquet(DIRECTORIO / nombre)
    verificacion.append({
        "archivo": nombre,
        "lectura_correcta": True,
        "filas": len(recuperado),
        "columnas": recuperado.shape[1],
        "nulos_totales": int(recuperado.isna().sum().sum()),
    })
display(pd.DataFrame(verificacion).style.hide(axis="index").set_caption("Lectura desde Parquet"))

archivo,lectura_correcta,filas,columnas,nulos_totales
equipos.parquet,True,32,12,0
partidos.parquet,True,64,26,174
clasificacion.parquet,True,32,21,16


---

### Resultado

La exploración permitió identificar la estructura real de cada endpoint. Las
respuestas anidadas se transformaron en tablas planas, tipadas, sin duplicados
y listas para el análisis. El siguiente paso se encuentra en `analisis.ipynb`,
que trabaja exclusivamente con los tres Parquet generados aquí.